# Clase 9 · Notebook 02
# Análisis de sentimiento con una RNN (Embedding + LSTM)

**Introducción al Deep Learning — Módulo 4, Unidad 1: Redes recurrentes y NLP**

Construiremos una red recurrente **many-to-one** que lee una frase y predice si es **positiva** o **negativa**.
Es el ejemplo clásico de NLP con redes recurrentes:

`Texto → Embedding → LSTM → Dense (sigmoide)`

Usamos un pequeño conjunto de frases en español para que entrene rápido y sin descargas.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Un pequeño conjunto de frases etiquetadas

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
tf.random.set_seed(42); np.random.seed(42)

positivas = [
    "me encanta esta pelicula es maravillosa",
    "que gran experiencia la recomiendo totalmente",
    "el servicio fue excelente y muy rapido",
    "un producto fantastico estoy muy contento",
    "comida deliciosa y trato amable",
    "una historia preciosa me hizo feliz",
    "perfecto funciona genial lo adoro",
    "calidad increible volvere a comprar",
    "el hotel era comodo y bonito",
    "me ha gustado mucho gracias",
]
negativas = [
    "odio esta pelicula es horrible y aburrida",
    "una experiencia pesima no la recomiendo",
    "el servicio fue lento y malo",
    "un producto defectuoso estoy muy decepcionado",
    "comida fria y trato desagradable",
    "una historia terrible me aburrio mucho",
    "no funciona es un desastre lo odio",
    "calidad penosa no volvere a comprar",
    "el hotel era sucio e incomodo",
    "no me ha gustado nada fatal",
]
textos = positivas + negativas
etiquetas = np.array([1]*len(positivas) + [0]*len(negativas), dtype="float32")
print("Frases:", len(textos), "| positivas=1, negativas=0")

## 2. Vectorizar el texto

`TextVectorization` aprende el vocabulario y convierte cada frase en una secuencia de enteros de longitud fija.

In [ ]:
vectorizar = tf.keras.layers.TextVectorization(output_mode="int", output_sequence_length=8)
vectorizar.adapt(textos)
VOCAB = len(vectorizar.get_vocabulary())
X = vectorizar(np.array(textos)).numpy()
print("Tamaño del vocabulario:", VOCAB)
print("Ejemplo:", textos[0])
print("  ->", X[0])

## 3. La red: Embedding + LSTM

- **Embedding**: convierte cada palabra (id) en un vector denso.
- **LSTM**: recorre la secuencia manteniendo memoria.
- **Dense (sigmoide)**: da un valor 0-1 (negativo ↔ positivo).

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input

modelo = Sequential([
    Input(shape=(8,)),
    Embedding(input_dim=VOCAB, output_dim=16),
    LSTM(16),
    Dense(1, activation="sigmoid"),
])
modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.summary()

## 4. Entrenar

In [ ]:
historia = modelo.fit(X, etiquetas, epochs=60, verbose=0)
print("Exactitud final en entrenamiento: %.0f %%" % (historia.history["accuracy"][-1] * 100))

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(historia.history["loss"], color="#000F9F")
plt.title("Pérdida durante el entrenamiento"); plt.xlabel("Época"); plt.ylabel("loss")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. Probar con frases nuevas

El modelo no ha visto estas frases. Veamos qué sentimiento les asigna.

In [ ]:
nuevas = [
    "me encanta el trato fue maravilloso",
    "una pelicula horrible y muy aburrida",
    "el producto es fantastico lo recomiendo",
    "servicio lento y comida fria",
]
probs = modelo.predict(vectorizar(np.array(nuevas)).numpy(), verbose=0).ravel()
for frase, pr in zip(nuevas, probs):
    sent = "POSITIVO" if pr >= 0.5 else "NEGATIVO"
    print(f"[{sent}  {pr:.2f}]  {frase}")

La red, partiendo de muy pocos ejemplos, aprende a separar frases positivas de negativas leyendo la
**secuencia** de palabras (no solo su presencia). Con un corpus grande (p. ej. reseñas reales) y más
épocas, la calidad mejora mucho.

## Experimenta tú

1. Escribe tus propias frases en la lista `nuevas` y observa la predicción.
2. Cambia `LSTM` por `GRU` y compara.
3. Aumenta el tamaño del `Embedding` o de la `LSTM`.

## Resumen

- Una RNN **many-to-one** para sentimiento: `Embedding → LSTM → Dense (sigmoide)`.
- El **Embedding** representa palabras como vectores; la **LSTM** recorre la secuencia con memoria.
- Salida `sigmoide` + pérdida `binary_crossentropy` para clasificar positivo/negativo.

Con esto cerramos la unidad de redes recurrentes y NLP. En la próxima clase veremos el **aprendizaje por refuerzo**.